## Interactive viz—bokeh

![](img/bokeh%20logo.svg)

---

### Get prepared

#### Installations

`py -m pip install bokeh`  
`py -m pip install jupyter_bokeh`  
`py -m pip install selenium`  
geckodriver.exe, add to Path

#### Imports

In [ ]:
import math
import networkx as nx
from bokeh.plotting import figure, show
from bokeh.models import GraphRenderer, Ellipse, StaticLayoutProvider
from bokeh.models import ColumnDataSource, LabelSet
from bokeh.palettes import Spectral8
from bokeh.palettes import Category20_20
from bokeh.plotting import figure, from_networkx, show
from bokeh.models import (BoxSelectTool, HoverTool, MultiLine,
                          NodesAndLinkedEdges, EdgesAndLinkedNodes, Plot, NodesAndAdjacentNodes, Range1d, Scatter, TapTool)
from bokeh.palettes import Spectral4
from bokeh.io import output_notebook, output_file, export_svg
import json
import pandas as pd

---

### NetworkX integration

Bokeh integrates the NetworkX package so you can quickly plot network graphs. The bokeh.plotting.from_networkx convenience method accepts a networkx.Graph object and a NetworkX layout method and returns a configured instance of the GraphRenderer model.


#### Zachary’s karate club graph

Here is how the networkx.spring_layout method lays out the “Zachary’s karate club graph” data set built into NetworkX:

In [ ]:
G = nx.desargues_graph() # always 20 nodes
# drawing area
p = figure(x_range=(-2, 2), y_range=(-2, 2),
           x_axis_location=None, y_axis_location=None,
           tools="hover", tooltips="index: @index")
p.grid.grid_line_color = None

# node positions from graphviz
graph = from_networkx(G, nx.spring_layout, scale=1.8, center=(0,0))
p.renderers.append(graph)

# nodes 
# Add some new columns to the node renderer data source
graph.node_renderer.data_source.data['index']  = list(range(len(G)))
graph.node_renderer.data_source.data['colors'] = Category20_20

# edges
graph.node_renderer.glyph.update(size=20, fill_color="colors")

show(p)

#### Node and edge attributes

The `from_networkx` method converts node and edge attributes of the NetworkX package for use with `node_renderer` and `edge_renderer` of the GraphRenderer model.

For example, “Zachary’s karate club graph” data set has a node attribute named “club”. You can hover this information with node attributes converted with the `from_networkx` method. You can also use node and edge attributes for color information.

Here is an example of a graph that hovers node attributes and changes colors with edge attributes:

In [ ]:
# NetworkX graph with additionaledge attributes
G = nx.karate_club_graph()
SAME_CLUB_COLOR, DIFFERENT_CLUB_COLOR = "darkgrey", "red"
edge_attrs = {}
for start_node, end_node, _ in G.edges(data=True):
    edge_color = SAME_CLUB_COLOR if G.nodes[start_node]["club"] == G.nodes[end_node]["club"] else DIFFERENT_CLUB_COLOR
    edge_attrs[(start_node, end_node)] = edge_color
nx.set_edge_attributes(G, edge_attrs, "edge_color")

# drawing area
plot = figure(width=1200, height=1200, x_range=(-1.2, 1.2), y_range=(-1.2, 1.2),
              x_axis_location=None, y_axis_location=None, toolbar_location=None,
              title="Graph Interaction Demo", background_fill_color="white",
              tooltips="index: @index, club: @club")
plot.grid.grid_line_color = None

# node positions from NetworkX
graph_renderer = from_networkx(G, nx.spring_layout, scale=1, center=(0, 0))

# nodes
graph_renderer.node_renderer.glyph = Scatter(size=15, fill_color="lightblue")

# edges
graph_renderer.edge_renderer.glyph = MultiLine(line_color="edge_color", line_alpha=1, line_width=2)

# render the graph
plot.renderers.append(graph_renderer)
show(plot)

#### Export to svg and show in notebook

In [ ]:
# NetworkX graph with additionaledge attributes
G = nx.karate_club_graph()
SAME_CLUB_COLOR, DIFFERENT_CLUB_COLOR = "darkgrey", "red"
edge_attrs = {}
for start_node, end_node, _ in G.edges(data=True):
    edge_color = SAME_CLUB_COLOR if G.nodes[start_node]["club"] == G.nodes[end_node]["club"] else DIFFERENT_CLUB_COLOR
    edge_attrs[(start_node, end_node)] = edge_color
nx.set_edge_attributes(G, edge_attrs, "edge_color")

# drawing area
plot = figure(width=1200, height=1200, x_range=(-1.2, 1.2), y_range=(-1.2, 1.2),
              x_axis_location=None, y_axis_location=None, toolbar_location=None,
              title="Graph Interaction Demo", background_fill_color="white",
              tooltips="index: @index, club: @club")
plot.grid.grid_line_color = None

# node positions from NetworkX
graph_renderer = from_networkx(G, nx.spring_layout, scale=1, center=(0, 0))

# nodes
graph_renderer.node_renderer.glyph = Scatter(size=15, fill_color="lightblue")

# edges
graph_renderer.edge_renderer.glyph = MultiLine(line_color="edge_color", line_alpha=1, line_width=2)

# render the graph
output_notebook()
plot.renderers.append(graph_renderer)
show(plot)
plot.output_backend = "svg"
export_svg(plot, filename="graphs/zachary_karate-club.svg")

#### Interaction policies

You can configure the selection or inspection behavior of graphs by setting the selection_policy and inspection_policy attributes of the GraphRenderer. These policy attributes accept a special GraphHitTestPolicy model instance.

For example, setting selection_policy to NodesAndLinkedEdges() lets you select a node and all associated edges. Similarly, setting inspection_policy to EdgesAndLinkedNodes() lets you inspect the "start" and "end" nodes of an edge by hovering over it with the HoverTool. NodesAndAdjacentNodes() lets you inspect a node and all other nodes connected to it by a graph edge.

You can customize the selection_glyph, nonselection_glyph, and/or hover_glyph attributes of the edge and node sub-renderers to add dynamic visual elements to your graph interactions.

Nodes and linked edges

In [ ]:
G = nx.karate_club_graph()

plot = Plot(width=800, height=800, x_range=Range1d(-1.1, 1.1), y_range=Range1d(-1.1, 1.1))
plot.title.text = "Show nodes and linked edges on hover"

plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

graph_renderer = from_networkx(G, nx.circular_layout, scale=1, center=(0, 0))

scatter_glyph = Scatter(size=15, fill_color=Spectral4[0])
graph_renderer.node_renderer.glyph = scatter_glyph
graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[2])
graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[1])

ml_glyph = MultiLine(line_color="#CCCCCC", line_alpha=0.8, line_width=1)
graph_renderer.edge_renderer.glyph = ml_glyph
graph_renderer.edge_renderer.selection_glyph = ml_glyph.clone(line_color=Spectral4[2], line_alpha=1)
graph_renderer.edge_renderer.hover_glyph = ml_glyph.clone(line_color=Spectral4[3], line_width=2)

graph_renderer.selection_policy = NodesAndLinkedEdges()
graph_renderer.inspection_policy = NodesAndLinkedEdges()

plot.renderers.append(graph_renderer)

show(plot)

Edges and linked nodes

In [ ]:
G = nx.karate_club_graph()

plot = Plot(width=1200, height=1200, x_range=Range1d(-1.1, 1.1), y_range=Range1d(-1.1, 1.1))
plot.title.text = "Show edges and linked nodes on hover"

plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

graph_renderer = from_networkx(G, nx.circular_layout, scale=1, center=(0, 0))

scatter_glyph = Scatter(size=15, fill_color=Spectral4[0])
graph_renderer.node_renderer.glyph = scatter_glyph
graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[2])
graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[1])

ml_glyph = MultiLine(line_color="#CCCCCC", line_alpha=0.8, line_width=1)
graph_renderer.edge_renderer.glyph = ml_glyph
graph_renderer.edge_renderer.selection_glyph = ml_glyph.clone(line_color=Spectral4[2], line_alpha=1)
graph_renderer.edge_renderer.hover_glyph = ml_glyph.clone(line_color=Spectral4[3], line_width=1)

graph_renderer.selection_policy = EdgesAndLinkedNodes()
graph_renderer.inspection_policy = EdgesAndLinkedNodes()

plot.renderers.append(graph_renderer)

show(plot)

Nodes and adjacent nodes

In [ ]:
G = nx.karate_club_graph()

plot = Plot(width=1200, height=1200, x_range=Range1d(-1.1, 1.1), y_range=Range1d(-1.1, 1.1))
plot.title.text = "Show nodes and adjacent nodes on hover"

plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

graph_renderer = from_networkx(G, nx.circular_layout, scale=1, center=(0, 0))

scatter_glyph = Scatter(size=15, fill_color=Spectral4[0])
graph_renderer.node_renderer.glyph = scatter_glyph
graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[2])
graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[1])

ml_glyph = MultiLine(line_color="#CCCCCC", line_alpha=0.8, line_width=1)
graph_renderer.edge_renderer.glyph = ml_glyph
graph_renderer.edge_renderer.selection_glyph = ml_glyph.clone(line_color=Spectral4[2], line_alpha=1)
graph_renderer.edge_renderer.hover_glyph = ml_glyph.clone(line_color=Spectral4[2], line_width=1)

graph_renderer.selection_policy = NodesAndAdjacentNodes()
graph_renderer.inspection_policy = NodesAndAdjacentNodes()

plot.renderers.append(graph_renderer)

show(plot)

#### Set node color attribute and combine with hover and select

In [ ]:
G = nx.karate_club_graph()

# set edge color based on whether the nodes at either end of the edge belong to the same club or not
SAME_CLUB_COLOR, DIFFERENT_CLUB_COLOR = "darkgrey", "red"
node_attrs = {}
for node, _ in G.nodes(data=True):
    node_color = SAME_CLUB_COLOR if G.degree[node] <= 4 else DIFFERENT_CLUB_COLOR
    node_attrs[node] = node_color
nx.set_node_attributes(G, node_attrs, "node_color")

# drawing area
plot = figure(width=1200, height=1200, x_range=(-1.2, 1.2), y_range=(-1.2, 1.2),
              x_axis_location=None, y_axis_location=None, toolbar_location=None,
              title="Node color attribute", tooltips="index: @index, club: @club")
plot.grid.grid_line_color = None

plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

# get node placement from NetworkX spring layout
graph_renderer = from_networkx(G, nx.spring_layout, scale=1, center=(0, 0))

#nodes
scatter_glyph = Scatter(size=15, fill_color="node_color")
graph_renderer.node_renderer.glyph = scatter_glyph
graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[3])
graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[3])

#edges
graph_renderer.edge_renderer.glyph = MultiLine(line_color="lightblue", line_alpha=1, line_width=2)
graph_renderer.selection_policy = NodesAndAdjacentNodes()
graph_renderer.inspection_policy = NodesAndAdjacentNodes()

# render the graph
plot.renderers.append(graph_renderer)
show(plot)

Combine with graph export

In [ ]:
G = nx.karate_club_graph()

# set edge color based on whether the nodes at either end of the edge belong to the same club or not
SAME_CLUB_COLOR, DIFFERENT_CLUB_COLOR = "darkgrey", "red"

node_attrs = {}
for node, _ in G.nodes(data=True):
    node_color = SAME_CLUB_COLOR if G.degree[node] <= 4 else DIFFERENT_CLUB_COLOR
    node_attrs[node] = node_color
nx.set_node_attributes(G, node_attrs, "node_color")
#print(G.nodes(data=True))

# setup drawing area
plot = figure(width=1200, height=1200, x_range=(-1.2, 1.2), y_range=(-1.2, 1.2),
              x_axis_location=None, y_axis_location=None, toolbar_location=None,
              title="Node color attribute", tooltips="index: @index, club: @club")
plot.grid.grid_line_color = None

#plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

# get node placement from NetworkX spring layout
graph_renderer = from_networkx(G, nx.spring_layout, scale=1, center=(0, 0))

#set node renderer
graph_renderer.node_renderer.glyph = Scatter(size=15, fill_color="node_color")
#graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[3])
#graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[3])

#set edge renderer
graph_renderer.edge_renderer.glyph = MultiLine(line_color="lightblue", line_alpha=1, line_width=2)

#graph_renderer.selection_policy = NodesAndAdjacentNodes()
#graph_renderer.inspection_policy = NodesAndAdjacentNodes()

# render the graph
plot.renderers.append(graph_renderer)

show(plot)

---

### Viz for Pres Inaug Addr—matplotlib coordinates

In [ ]:
# Reconstruct the graph
with open('graphs/USPresInaugAddr_0.14.json', 'r', encoding='utf-8') as f:
    InaugAddr_json = json.load(f)
G = nx.node_link_graph(InaugAddr_json)
print(G)
#print (G.nodes(data=True))

f=3
p = figure(width=1500, height=1500,x_range=(-f/2,f/2), y_range=(-f/2,f/2),
           x_axis_location=None, y_axis_location=None,
           tools="hover", tooltips="index: @index")
p.grid.grid_line_color = None

graph = from_networkx(G, nx.spring_layout, scale=5, center=(0,0))
p.renderers.append(graph)

# Add some new columns to the node renderer data source
#graph.node_renderer.data_source.data['index'] = list(range(len(G)))
graph.node_renderer.glyph = Scatter(size=10, fill_color="lightblue")

show(p)

---

### Viz for Pres Inaug Addr—Graphviz coordinates

#### Read graph

In [7]:
# Reconstruct the graph
with open('graphs/USPresInaugAddr_0.14.json', 'r', encoding='utf-8') as f:
    InaugAddr_json = json.load(f)
G = nx.node_link_graph(InaugAddr_json)
print(G)

Graph with 153 nodes and 604 edges


#### Create dict for node positions from .dot.json file

In [6]:
nodes_dict = {} # dict for node positions, serving as layout provider for Bokeh graph rendering
with open('graphs/USPresInaugAddr_0.14.dot.json', 'r', encoding='utf-8') as json_data:
    data = json.load(json_data)
for node in data['objects']:
    pos_raw = node['pos']
    x_str, y_str = pos_raw.strip('"').split(',')
    x = float(x_str)
    y = float(y_str)
    nodes_dict[node['name']] = (x, y)
print(nodes_dict)

{'arrive': (7476.3, 93.984), 'incur': (7956.7, 231.09), 'distinguished': (7925.0, 1072.8), 'willingly': (6415.3, 1427.2), 'oath': (6389.5, 926.08), 'magistrate': (7106.0, 696.7), 'upbraiding': (7699.2, 1533.0), 'official': (6671.6, 1844.3), 'violate': (7191.4, 1590.9), 'instance': (6592.6, 480.21), 'ceremony': (7096.6, 2121.6), 'knowingly': (7618.8, 664.06), 'execution': (8339.9, 1548.0), 'punishment': (6904.7, 1166.9), 'previous': (8050.0, 1947.0), 'witness': (8289.2, 591.85), 'injunction': (7589.7, 2131.8), 'repose': (8458.3, 1064.6), 'entertain': (6992.6, 164.49), 'government': (3156.6, 3802.0), 'inhabitant': (2210.9, 4808.8), 'economic': (3376.9, 4797.7), 'freedom': (3969.2, 2549.1), 'citizen': (1971.0, 2372.3), 'constitution': (1377.2, 2559.7), 'cuba': (1679.3, 4202.2), 'great': (2849.9, 5270.7), 'invasion': (2345.3, 5759.5), 'nuclear': (3178.6, 2928.2), 'partisan': (2403.7, 1599.0), 'pleasing': (3803.1, 3070.2), 'disaster': (3457.8, 4517.0), 'hero': (3745.2, 4591.0), 'public': (2

#### Create color palette

In [8]:
from collections import Counter
with open(f"misc/meta.json", 'r', encoding='utf-8') as f:
    meta = json.load(f)
print(meta)
colors = []
for _, att in G.nodes(data=True):
    xx = att['used_in']
    y = [meta[x]['party'] for x in xx]
    z = dict(Counter(y))
    if 'Republikaner' in z and 'Demokrat' in z:
        colors.append('#D8BFD8')
    elif 'Republikaner' in z:
        colors.append('#E9141D')
    elif 'Demokrat' in z:
        colors.append('#0015BC')
    else:
        colors.append('#ADD8E6')
print(len(colors), colors[20:30])

{'01_washington_1789': {'president': 'George Washington', 'party': 'parteilos', 'date': '1789-04-30', 'year': '1789'}, '02_washington_1793': {'president': 'George Washington', 'party': 'parteilos', 'date': '1793-03-04', 'year': '1793'}, '03_adams_john_1797': {'president': 'John Adams', 'party': 'Föderalist', 'date': '1797-03-04', 'year': '1797'}, '04_jefferson_1801': {'president': 'Thomas Jefferson', 'party': 'Republikaner', 'date': '1801-03-04', 'year': '1801'}, '05_jefferson_1805': {'president': 'Thomas Jefferson', 'party': 'Republikaner', 'date': '1805-03-04', 'year': '1805'}, '06_madison_1809': {'president': 'James Madison', 'party': 'Republikaner', 'date': '1809-03-04', 'year': '1809'}, '07_madison_1813': {'president': 'James Madison', 'party': 'Republikaner', 'date': '1813-03-04', 'year': '1813'}, '08_monroe_1817': {'president': 'James Monroe', 'party': 'Republikaner', 'date': '1817-03-04', 'year': '1817'}, '09_monroe_1821': {'president': 'James Monroe', 'party': 'Republikaner', 

#### Create size list

In [9]:
def node_size(param=None):
    return 16.0 + 8*math.sqrt(param)

sizes = [node_size(doc_frq) for doc_frq in list(nx.get_node_attributes(G, 'doc_frq').values())]
print (sizes)

[33.88854381999832, 38.62741699796952, 45.93325909419153, 46.983866769659336, 41.29822128134704, 32.0, 32.0, 42.5329983228432, 29.856406460551018, 38.62741699796952, 62.647615158762406, 45.93325909419153, 29.856406460551018, 40.0, 41.29822128134704, 24.0, 41.29822128134704, 27.31370849898476, 46.983866769659336, 75.3295878967653, 35.59591794226542, 76.398675482166, 75.86651818838305, 27.31370849898476, 65.95998398718719, 40.0, 67.84592558726288, 33.88854381999832, 27.31370849898476, 70.25863986500215, 75.86651818838305, 37.166010488516726, 72.0, 60.54211490264017, 56.0, 48.984845004941285, 58.33202097703345, 74.78775382679628, 65.3153120237518, 57.569219381653056, 48.0, 48.984845004941285, 27.31370849898476, 64.0, 49.94112549695428, 46.983866769659336, 24.0, 70.25863986500215, 38.62741699796952, 45.93325909419153, 41.29822128134704, 65.3153120237518, 27.31370849898476, 27.31370849898476, 24.0, 49.94112549695428, 33.88854381999832, 54.36665218650175, 35.59591794226542, 33.88854381999832

#### Color brightness

In [10]:
def brightness_from_rgb(rgb):
    rgb = rgb.replace('#', '0x') 
    rgb = int(rgb, 16)
    r = (rgb >> 16) & 0xFF
    g = (rgb >> 8) & 0xFF
    b = rgb & 0xFF
    brightness = math.sqrt(0.299 * math.pow(r, 2) + 0.587 * math.pow(g, 2) + 0.114 * math.pow(b, 2))
    if brightness <= 130.0:
        return "dark"
    else:
        return "light"

#### Create label set

In [11]:
source = ColumnDataSource(data=dict(
    x=[float(coor[0]) for coor in nodes_dict.values()],
    y=[float(coor[1]) for coor in nodes_dict.values()],
    name=[name for name in nodes_dict.keys()],
    text=[name if G.nodes[name]['doc_frq'] > 40 else "" for name in nodes_dict.keys()],
    text_color=['black' if brightness_from_rgb(colors[list(G.nodes()).index(name)]) == 'light' else 'white' for name in nodes_dict.keys()],
))
labels = LabelSet(x='x', y='y', text='text', text_align='center', text_baseline='middle', text_color='text_color', source=source)


#### Create bokeh viz

In [ ]:
# drawing area
plot = figure(title="Using graphviz coordinates", 
            width=1800, height=1600, 
            margin=(80,80,80,80),
            x_range=(-200,8800), y_range=(-1000,8000), 
            x_axis_location=None, y_axis_location=None,
            tools="hover",  tooltips="""name: @index<br>used_in: @used_in""",
            toolbar_location=None)
plot.grid.grid_line_color = None

# graph
graph = GraphRenderer()

# nodes
graph.node_renderer.data_source.data = dict(
    index=list(G.nodes()),
    used_in=list(nx.get_node_attributes(G, 'used_in').values()), 
    fill_color=colors,
    size=sizes)
graph.node_renderer.glyph = Scatter(size="size", fill_color="fill_color")

# edges
graph.edge_renderer.data_source.data = dict(
    start=[edge[0] for edge in G.edges()],   
    end  =[edge[1] for edge in G.edges()])
graph.edge_renderer.glyph = MultiLine(line_color="darkgray", line_alpha=1, line_width=1)

# node positions from graphviz
graph.layout_provider = StaticLayoutProvider(graph_layout=nodes_dict)

# render the graph
plot.renderers.append(graph)
plot.add_layout(labels)
output_notebook()
plot.output_backend = "svg"
export_svg(plot, filename="graphs/USPresInaugAddr_0.14.bokeh.svg")
show(plot)

#### Add interactivity

Select node and show its neighbours

In [15]:
# drawing area
plot = figure(title="Using graphviz coordinates", 
            width=1800, height=1600, 
            margin=(10,10,10,10),
            x_range=(-200,8800), y_range=(-1000,8000), 
            x_axis_location=None, y_axis_location=None
            )
plot.grid.grid_line_color = None
plot.add_tools(HoverTool(tooltips="""name: @index<br>used_in: @used_in"""), TapTool(), BoxSelectTool())

# graph
graph = GraphRenderer()

# nodes
graph.node_renderer.data_source.data = dict(
    index=list(G.nodes()),
    used_in=list(nx.get_node_attributes(G, 'used_in').values()), 
    fill_color=colors,
    size=sizes)
shape = Scatter(size="size", fill_color="fill_color")
graph.node_renderer.glyph = shape
graph.node_renderer.selection_glyph = shape.clone(fill_color="green")
graph.node_renderer.hover_glyph = shape.clone(fill_color="green")


# edges
graph.edge_renderer.data_source.data = dict(
    start=[edge[0] for edge in G.edges()],   
    end  =[edge[1] for edge in G.edges()])
graph.edge_renderer.glyph = MultiLine(line_color="darkgray", line_alpha=1, line_width=1)

# node positions from graphviz
graph.layout_provider = StaticLayoutProvider(graph_layout=nodes_dict)

# render the graph
plot.renderers.append(graph)
plot.add_layout(labels)
output_notebook()
#plot.output_backend = "svg"
#export_svg(plot, filename="graphs/USPresInaugAddr_0.14.bokeh.svg")
show(plot)

Loading BokehJS ...

---

### Viz for Netflix co-actor graph

#### Read graph and its node positions

In [ ]:
with open('graphs/co-actor_graph.json', 'r', encoding='utf-8') as f:
    InaugAddr_json = json.load(f)
C = nx.node_link_graph(InaugAddr_json)
print(C)

In [ ]:
with open('graphs/co-actor_graph_positions.json', 'r', encoding='utf-8') as f:
    pos_json = json.load(f)
co_actor_pos = dict(pos_json)
print(co_actor_pos)

#### Prepare for bokeh

In [ ]:
# compute degree and assign colors by degree buckets
degree_by_node = dict(C.degree())

# simple degree-based color mapping
def degree_color(degree):
    if degree <= 2:
        return '#a6cee3'
    elif degree <= 5:
        return '#1f78b4'
    elif degree <= 25:
        return '#b2df8a'
    elif degree <= 65:
        return '#33a02c'
    else:
        return '#e31a1c'

node_colors = [degree_color(degree_by_node[node]) for node in C.nodes()]
node_degrees = [degree_by_node[node] for node in C.nodes()]

# drawing area
plot = figure(title="Netflix co-actor graph using matplotlib coordinates", 
            min_width=1800, min_height=1800, 
            margin=(80,80,80,80),
            x_range=(-1.1,1.1), y_range=(-1.,1), 
            x_axis_location=None, y_axis_location=None,
            tools="hover", tooltips="@index, degree: @degree",   
            toolbar_location=None)
plot.grid.grid_line_color = None

# graph
graph = GraphRenderer()

# nodes
graph.node_renderer.data_source.data = dict(
    index=list(C.nodes()),
    degree=node_degrees,
    fill_color=node_colors,
)
graph.node_renderer.glyph = Scatter(size=15.0, fill_color="fill_color")

# edges
graph.edge_renderer.data_source.data = dict(
    start=[edge[0] for edge in C.edges()],   
    end  =[edge[1] for edge in C.edges()])
graph.edge_renderer.glyph = MultiLine(line_color="darkgray", line_alpha=1, line_width=1)

# node positions from graphviz
graph.layout_provider = StaticLayoutProvider(graph_layout=co_actor_pos)

# render the graph
plot.renderers.append(graph)
#plot.add_layout(labels)
output_notebook()
plot.output_backend = "svg"
export_svg(plot, filename="graphs/co-actor_graph.bokeh.svg")
show(plot)